        # 04 感度分析

        - 目的：貯蓄率・収益率・生活費・FIRE 目標が、FIRE 達成年齢にどれだけ影響するかを調べる
        - 計算：`optimization.py`（全探索 $O(K_1 K_2 T)$）

> 架空のデータによる**教育用シミュレーション**です。将来の投資収益の予測・保証や、投資・金融商品の推奨ではありません。金額は**万円**。税・社会保険料・インフレ・手数料・年金は考慮していません。

## 準備：実験環境（最初に必ず実行）

F2 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F2 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F2 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F2 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                               # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f2-financial-independence"   # リポジトリの中の F2 フォルダ（ルートが F2 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f2_root(p):
    return (Path(p) / "financial_model.py").exists() and (Path(p) / "fire_analysis.py").exists()


def find_f2_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F2 のルート
        if is_f2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f2_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f2_upload")
    for hit in glob.glob("/content/f2_upload/**/financial_model.py", recursive=True):
        if is_f2_root(Path(hit).parent):
            return Path(hit).parent
    return None


F2_ROOT = find_f2_root()
if F2_ROOT is None:
    raise RuntimeError("F2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F2_ROOT)
if str(F2_ROOT) not in sys.path:
    sys.path.insert(0, str(F2_ROOT))
if not (F2_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。fire_analysis.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import financial_model as fm
import fire_analysis as fa
import retirement_model as rm
import optimization as op
import fire_charts as fc

fc.setup_style()                     # 文字を大きくするなら fc.setup_style(1.3)
scenarios, settings = fm.load_scenarios()
named = {s["name"]: s for s in scenarios}
END_ACC = settings["accumulation_end_age"]      # 資産形成の計算終了年齢（80）
END_RET = settings["retirement_end_age"]        # リタイア後の計算終了年齢（100）
W = settings["withdrawal_rate"]                 # 取り崩し率（4%、教育用の仮定）

SAVE_FIGURES = False                 # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", fc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: (("not reached" if k == "fire_age" else "-") if r.get(k) is None else f.format(r[k]))
             for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F2_ROOT:", F2_ROOT)
print("シナリオ:", [s["label"] for s in scenarios], f"/ 取り崩し率 {W:.0%} / 資産形成 {END_ACC} 歳まで / リタイア後 {END_RET} 歳まで")

## 1. 貯蓄率 × 収益率のヒートマップ

所得 400 万円を固定（生活費 = 所得 × (1 − 貯蓄率)）。**斜線の N/A は 80 歳までに未達成**で、数値 0 や 80 とは別に表示します。

In [ ]:
#@title グラフを表示（コードは隠しています）
base = named["standard"]
rates = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
returns = [0.0, 0.02, 0.04, 0.06, 0.08]
matrix = op.grid_fire_age(base, rates, returns, settings, keep="income")
show(fc.plot_heatmap(matrix, rates, returns, end_age=END_ACC), "04_heatmap")
print(f"組合せ {len(rates)} × {len(returns)} = {len(rates) * len(returns)} 通り × {END_ACC - base['start_age']} 年")

## 2. 生活費と FIRE 達成年齢（所得固定 / 貯蓄率固定）

In [ ]:
#@title グラフを表示（コードは隠しています）
expenses = list(range(160, 381, 20))
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
for ax, keep, title in [(axes[0], "income", "Income fixed (400)"), (axes[1], "savings_rate", "Savings rate fixed (20%)")]:
    rows = op.sweep_expenses(base, expenses, settings, keep=keep)
    fc.plot_fire_age(expenses, [r["fire_age"] for r in rows], "annual expenses", title, END_ACC, xfmt="{}", ax=ax)
    ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
show(fig, "04_expenses")

## 3. 1 つずつ変えたときの影響（トルネード図）

基準（Standard）から 1 つの仮定だけを変え、FIRE 達成年齢の変化を比べます。**変える幅は仮定です**（TODO：チームで決める）。

In [ ]:
#@title グラフを表示（コードは隠しています）
ref = op.evaluate_scenario(base, settings)["fire_age"]

def fire_with(**ch):
    sets = dict(settings, **ch.pop("_settings", {}))
    return op.evaluate_scenario(fm.with_changes(base, **ch), sets)["fire_age"]

sens = [
    {"label": "savings rate 15% / 25%", "low": fire_with(savings_rate=0.15, annual_expenses=340),
     "high": fire_with(savings_rate=0.25, annual_expenses=300)},
    {"label": "return 2% / 4%", "low": fire_with(annual_return=0.02), "high": fire_with(annual_return=0.04)},
    {"label": "withdrawal rate 3.5% / 4.5%", "low": fire_with(_settings={"withdrawal_rate": 0.035}),
     "high": fire_with(_settings={"withdrawal_rate": 0.045})},
    {"label": "initial assets 0 / 500", "low": ref, "high": fire_with(initial_assets=500)},
]
show(fc.plot_tornado(sens, ref, "Sensitivity of FIRE age (Standard)"), "04_tornado")
table(sens, [("label", "assumption", "{}"), ("low", "lowered", "{}"), ("high", "raised", "{}")])

## 4. 条件別の比較表

In [ ]:
cases = {
    "Standard": base,
    "savings 30%": fm.with_changes(base, savings_rate=0.3, annual_expenses=280),
    "return 5%": fm.with_changes(base, annual_return=0.05),
    "expenses 280 (target only)": fm.with_changes(base, annual_expenses=280),
    "initial 1000": fm.with_changes(base, initial_assets=1000),
}
rows = []
for name, s in cases.items():
    r = op.evaluate_scenario(s, settings)
    r["case"] = name
    rows.append(r)
table(op.rank(rows, "fire_age", minimize=True),
      [("case", "case", "{}"), ("target", "FIRE target", "{:,.0f}"), ("fire_age", "FIRE age", "{}"),
       ("assets_at_report", "assets at 65", "{:,.0f}"), ("depletion_age", "depletion age", "{}")])
# TODO：自分たちの条件を cases に追加しよう

## 学生向け TODO・考察

- [ ] ヒートマップの範囲（貯蓄率・収益率）を変える
- [ ] トルネード図の変える幅を、根拠をもって決める（同じ幅？ 現実的な幅？）
- [ ] FIRE 達成年齢だけでなく、資産持続年数（`years_sustained`）でも感度分析する

**考察（チームで記入）**：TODO（どの仮定に最も敏感か、その仮定の不確実性はどのくらいか）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索・二分探索、第6章 挿入ソート）
- F2 リポジトリの `README.md`（モデルの仮定、4% ルールとその限界、計算量、モデルの限界と倫理的配慮）
- データ：`data/sample_scenarios.json`（この教材のための**架空のデータ**）